# Scraping Data Ulasan Google Play Store

**Nama**: Asep Haryana Saputra
**Proyek**: Analisis Sentimen (Minecraft)

## 1. Pendahuluan
Dalam notebook ini, kita akan melakukan **Web Scraping** untuk mengambil data ulasan pengguna dari aplikasi **Minecraft** (`com.mojang.minecraftpe`) di Google Play Store. 

Tujuannya adalah mengumpulkan dataset sebanyak **40.000 ulasan** yang nantinya akan digunakan untuk melatih model Analisis Sentimen.

### Alur Pengerjaan:
1.  **Instalasi Library**: Menginstall `google-play-scraper`.
2.  **Konfigurasi**: Menentukan ID aplikasi dan parameter scraping.
3.  **Proses Scraping**: Mengambil data secara bertahap (batch) agar progress terlihat.
4.  **Penyimpanan**: Menyimpan hasil ke format CSV.

## 2. Instalasi Library
Kita menggunakan library `google-play-scraper` yang merupakan library populer untuk mengambil data dari Google Play tanpa memerlukan API Key resmi.

In [ ]:
!pip install google-play-scraper pandas tqdm

## 3. Import Library & Konfigurasi
Di sini kita menyiapkan variabel-variabel utama seperti:
- `APP_ID`: ID unik aplikasi di Play Store.
- `TARGET_COUNT`: Jumlah data yang ingin diambil (40.000).
- `LANG` & `COUNTRY`: Bahasa (Inggris) dan lokasi (US) untuk mendapatkan ulasan global.

In [ ]:
import pandas as pd
from google_play_scraper import reviews, Sort
from tqdm import tqdm  # Library untuk progress bar
import time

# Konfigurasi Utama
APP_ID = 'com.mojang.minecraftpe'  # Minecraft Trial / Full
LANG = 'en'
COUNTRY = 'us'
TARGET_COUNT = 40000  # Target jumlah data
BATCH_SIZE = 1000     # Jumlah data yang diambil per request

print(f"Target Aplikasi: {APP_ID}")
print(f"Target Jumlah Data: {TARGET_COUNT}")

## 4. Proses Scraping dengan Progress Bar
Proses pengambilan data dilakukan dengan menggunakan `continuation_token`. Ini memungkinkan kita untuk mengambil data halaman demi halaman (pagination) dan menampilkan progress bar real-time menggunakan `tqdm`.

In [ ]:
def scrape_reviews_with_progress():
    reviews_all = []
    continuation_token = None
    
    # Inisialisasi Progress Bar
    with tqdm(total=TARGET_COUNT, desc="Mengambil Ulasan", unit="ulasan") as pbar:
        while len(reviews_all) < TARGET_COUNT:
            try:
                # Mengambil ulasan dalam batch
                result, continuation_token = reviews(
                    APP_ID,
                    count=BATCH_SIZE,
                    continuation_token=continuation_token,
                    lang=LANG,
                    country=COUNTRY,
                    sort=Sort.NEWEST
                )
                
                # Jika tidak ada hasil, hentikan loop
                if not result:
                    break
                
                # Tambahkan hasil ke list utama
                reviews_all.extend(result)
                
                # Update progress bar
                pbar.update(len(result))
                
                # Jika tidak ada token halaman selanjutnya, hentikan
                if not continuation_token:
                    break
                    
                # Optional: Sleep sebentar agar tidak terkena rate limit
                time.sleep(0.5)
                
            except Exception as e:
                print(f"Terjadi kesalahan: {e}")
                break
    
    # Potong jika melebihi target
    return reviews_all[:TARGET_COUNT]

# Jalankan fungsi scraping
raw_data = scrape_reviews_with_progress()

print(f"\nSelesai! Total ulasan yang didapatkan: {len(raw_data)}")

## 5. Penyimpanan dan Pemeriksaan Data
Data yang telah dikumpulkan dikonversi menjadi Pandas DataFrame dan disimpan ke dalam file CSV (`dataset_raw.csv`). Kita juga akan melihat sebaran skor (bintang) dari data yang didapat.

In [ ]:
# Konversi ke DataFrame
df = pd.DataFrame(raw_data)

# Simpan ke CSV
output_file = 'dataset_raw.csv'
df.to_csv(output_file, index=False)
print(f"Data berhasil disimpan ke: {output_file}")

# Menampilkan 5 data pertama
print("\nContoh Data:")
display(df.head())

# Menampilkan distribusi rating
print("\nDistribusi Rating (Bintang):")
print(df['score'].value_counts().sort_index())